# Olasılıksal Dil Modelleri

- Dilin yapısını ve düzenini anlamak ve modellemek için kullanılan istatistiksel yöntemlerdir.

Table of Contents

- [1. N-Gram Modelleri](#1.-N-Gram-Modelleri)
- [2. Gizli Markov Modelleri (Hidden Markov Models)](#2.-Gizli-Markov-Modelleri-(Hidden-Markov-Models):)
    - [2.a. Part of Speech (PoS)](#2.a.-Part-of-Speech-(PoS))
- [3. Maximum Enthropy Models](#3.-Maximum-Enthropy-Models:)

## 1. N-Gram Modelleri

- Bir dizideki (genellikle bir cümledeki) ardışık kelime veya karakter gruplarının olasılıklarını tahmin eder.
- *n* sayısı, gruptaki öğe sayısını belirtir.
    - <b><em>Uni-Gram (n = 1):</em></b> `This` `Is` `Big` `Data` `AI` `Book`
    - <b><em>Bi-Gram (n = 2):</em></b> `This is` `Is Big` `Big data` `Data AI` `AI Book`
    - <b><em>Tri-Gram (n = 3):</em></b> `This is big` `Is big data` `Big data AI` `Data AI Book`

- Bir cümlede bir kelimenin hangi olasılıkla diğer bir kelimeden sonra geleceğini tahmin etmek için kullanılır.
- Hesaplaması basit ve hızlıdır, yerel bağıntıları iyi yakalar
- Bağlam sınırlaması vardır, uzun mesafeleri bağımlılıkları yakalayamazlar. Yüksek veri gereksinimi vardır.

In [1]:
import pandas as pd
import nltk
from nltk.util import ngrams
from nltk.tokenize import word_tokenize

from collections import Counter

In [2]:
imdb_review_df = pd.read_csv("IMDB Dataset.csv")

In [28]:
import re
from nltk.corpus import stopwords
from bs4 import BeautifulSoup as bs
stop_words_eng = set(stopwords.words("english"))

def clean_text(text):

    # Gelen verinin type'ının kontrol edilmesi
    if not isinstance(text, str):
        text = str(text)
    
    # HTML tag'lerin temizlenmesi:
    text = bs(text, "html.parser").get_text()
    # Harf ya da boşluk olmayan tüm karakterlerin temizlenmesi:
    text = re.sub(r"[^A-Za-z\s]+", "", text)
    
    if not text: text = "text"
    # Text'in küçük harflere çevirilmesi
    text = text.lower()

    # Text temizliği:
    # Birden fazla tekrar eden boşluk karakterlerinin teke düşürülmesi:
    text = re.sub(r"\s{2, }", " ", text)
    # Birleşik yazılan stop-word'lerin ayırılması:
    text = re.sub(r"cannot", "can not", text)
    text = re.sub(r"im", "i am", text)

    # Stop-word'lerin temizlenmesi
    word_list = text.split()
    text = " ".join([word for word in word_list if word not in stop_words_eng])

    return text

In [60]:
reviews = imdb_review_df["review"].iloc[:3000].apply(clean_text)

In [61]:
tokens = [word_tokenize(sentence) for sentence in reviews]

In [62]:
bigrams = []
for token_list in tokens:
    bigrams.extend(list(ngrams(token_list, 2)))

In [63]:
bigrams_freq = Counter(bigrams)

In [75]:
bigrams_freq_df = pd.DataFrame({"Words": bigrams_freq.keys(), "Count": bigrams_freq.values()})
bigrams_freq_df.sort_values(by="Count", ascending=False).head(5)

,Words,Count
1231,"(ever, seen)",131
733,"(ive, seen)",129
9756,"(dont, know)",125
1333,"(special, effects)",124
3080,"(even, though)",120


In [76]:
trigrams = []
for token_list in tokens:
    trigrams.extend(list(ngrams(token_list, 3)))

trigrams_freq = Counter(trigrams)
trigrams_freq_df = pd.DataFrame({"Words": trigrams_freq.keys(), "Count": trigrams_freq.values()})
trigrams_freq_df.sort_values(by="Count", ascending=False).head(5)

,Words,Count
1230,"(ive, ever, seen)",57
9963,"(worst, movie, ever)",28
11828,"(movie, ive, ever)",18
9964,"(movie, ever, seen)",18
9796,"(one, worst, movies)",17


In [82]:
# "En iyi film" ve "En kötü film" olasılıkları:
bme_prob = trigrams_freq[("best", "movie", "ever")]/bigrams_freq[("movie", "ever")]
wme_prob = trigrams_freq[("worst", "movie", "ever")]/bigrams_freq[("movie", "ever")]

print(f"'____ movie ever'daki boşluğa 'best' ya da 'worst' gelme olasılıkları\n'best': {bme_prob*100}\n'worst': {wme_prob*100}")

'____ movie ever'daki boşluğa 'best' ya da 'worst' gelme olasılıkları
'best': 6.451612903225806
'worst': 45.16129032258064


## 2. Gizli Markov Modelleri (Hidden Markov Models):

- Bir dizi gözlemin (kelimeler ya da karakterler gibi) arkasında gizli bir durum dizisinin (örneğin dil bilgisel kategoriler) olduğu varsayımına dayanır.
- Her bir durmun belirli bir olasılıkla başka bir duruma geçeceğini ve her durumun belirli bir gözlemi üreteceğini varsayar.

![Hidden Markov Models Image](https://i.ytimg.com/vi/RWkHJnFj5rY/hq720.jpg?sqp=-oaymwEhCK4FEIIDSFryq4qpAxMIARUAAAAAGAElAADIQj0AgKJD&rs=AOn4CLB1YKn42svs95LecSRssvsHe1dN_w)

- **Kullanım:**
    - Konuşma tanıma
    - Dil modelleme
    - Parça etiketleme (Part-of-Speech tagging)
- **Avantajları:**
    - Bağlam modelleme
    - Verimli algoritmalar
- **Dezavantajları:**
    - Basitleştirici varsayımlar, bağımlılıkların yalnızca bir önceki durumdan geldiğini varsayar
    - Eğitim zorluğu, yüksek miktarda veriye ihtiyaç

### 2.a. Part-of-Speech (PoS)

In [2]:
import pandas as pd
import nltk
from nltk.tag import hmm
from nltk.corpus import conll2000

In [3]:
nltk.download("conll2000")

[nltk_data] Downloading package conll2000 to
[nltk_data]     C:\Users\mrtke\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\conll2000.zip.


True

In [4]:
train_data = conll2000.tagged_sents("train.txt")
test_data = conll2000.tagged_sents("test.txt")

In [5]:
trainer = hmm.HiddenMarkovModelTrainer()
hmm_tagger = trainer.train(train_data)

In [8]:
test_sentence = "I was always a curious child".split()
tags = hmm_tagger.tag(test_sentence)

C:\Users\mrtke\anaconda3\Lib\site-packages\nltk\tag\hmm.py:363: RuntimeWarning: overflow encountered in cast
  O[i, k] = self._output_logprob(si, self._symbols[k])


In [9]:
tags

[('I', 'PRP'),
 ('was', 'VBD'),
 ('always', 'RB'),
 ('a', 'DT'),
 ('curious', 'NN'),
 ('child', 'NN')]

## 3. Maximum Entropy Models:

- Bir olasılık dağılımı tahmin ederken mümkün olduğunca az varsayımda bulunmayı hedefler.

![MaxEnt Model Image](https://devopedia.org/images/article/225/6824.1570601351.png)
- **Kullanım:**
    - Özellikle sınıflandırma görevlerinde başarılıdır.
        - Örneğn bir cümlenin belirli bir sınıfa *(pozitif-negatif duygu vb.)* ait olma olsılığını tahmin edebilir.
- **Avantajlar:**
    - Esneklik
    - İyi genelleme
- **Dezavantajlar:**
    - Hesaplama maliyetleri
    - Özellik mühendisliği gerektirir, yani başarı çoğunlukla doğru ve anlamlı özelliklerin seçilmesine bağlıdır.

In [2]:
import pandas as pd
from nltk.classify import MaxentClassifier

In [5]:
sentiment_df = pd.read_csv("twitter_training.csv", names=["id","topic","sentiment","sentence"])

In [7]:
sentiment_df.drop(["id","topic"], axis=1, inplace=True)

In [29]:
training_data = sentiment_df["sentence"].apply(clean_text)

C:\Users\mrtke\AppData\Local\Temp\ipykernel_2148\200093467.py:13: MarkupResemblesLocatorWarning: The input passed in on this line looks more like a filename than HTML or XML.

If you meant to use Beautiful Soup to parse the contents of a file on disk, then something has gone wrong. You should open the file first, using code like this:

    filehandle = open(your filename)

You can then feed the open filehandle into Beautiful Soup instead of using the filename.

However, if you want to parse some data that happens to look like a filename, then nothing has gone wrong: you are using Beautiful Soup correctly, and this warning is spurious and can be filtered. To make this warning go away, run this code before calling the BeautifulSoup constructor:

    from bs4 import MarkupResemblesLocatorWarning
    import warnings

    warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)
    
  text = bs(text, "html.parser").get_text()


In [33]:
n = 10
print(f"Original sentence: {sentiment_df["sentence"].iloc[n]}\n Cleaned sentence: {training_data[n]}")

Original sentence: 2010 So I spent a few hours making something for fun. . . If you don't know I am a HUGE RhandlerR fan and Maya is one of my favorite characters. So I decided to make myself a wallpaper for my PC. . Here is the original image versus the creation I made :) Enjoy! pic.twitter.com/mLsI5wf9Jg
 Cleaned sentence: spent hours making something fun dont know huge rhandlerr fan maya one favorite characters decided make wallpaper pc original amage versus creation made enjoy pictwittercommlsiwfjg


In [46]:
train_data = [
    ({"love":True, "amazing":True, "garbage":False, "bussin":True, "goated":True},"positive"),
    ({"meh":True, "aight":True, "alright":False, "decent":True},"neutral"),
    ({"hate":True, "trash":True, "fantastic":False, "awful":True, "garbage":True},"negative"),
]

classifier = MaxentClassifier.train(train_data, max_iter=10)

  ==> Training (10 iterations)

      Iteration    Log Likelihood    Accuracy
      ---------------------------------------
             1          -1.09861        0.333
             2          -0.51083        1.000
             3          -0.33647        1.000
             4          -0.25131        1.000
             5          -0.20067        1.000
             6          -0.16705        1.000
             7          -0.14310        1.000
             8          -0.12516        1.000
             9          -0.11123        1.000
         Final          -0.10008        1.000


In [44]:
word_list = []

for i in range(0, len(train_data)):
    word_list.extend(train_data[i][0].keys())

word_list = list(set(word_list))

In [49]:
test = "tf2 is goated"
# Creating a dictionary of whether the words in the sentence exists in the given list
features = {word: (word in test.lower().split()) for word in word_list}

classifier.classify(features)

'positive'